# ViT vs D2NN (12层) CIFAR-10 对比（v2）

> 这版严格参考 `D2NN-single-layer-CIFAR10(FO).ipynb` 的训练思想：
> - 保留 `class DNN` 结构风格
> - 两阶段训练：先锁参数只训相位（+共享FC头），再解锁层间距联合微调
> - 损失函数保持 `MSELoss(reduction='sum')`
>
> 与原单层版本的改动：
> 1. detector 输出改为 **共享 FC head 输出**
> 2. `num_layers=12`

In [ ]:
import os
import math
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import vit_b_16
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# 中文字体显示修复（按可用字体自动选择）
plt.rcParams['font.sans-serif'] = ['SimHei', 'Noto Sans CJK SC', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print('✅ 依赖导入完成')

In [ ]:
# =========================
# 基础配置
# =========================
SEED = 42
BATCH_SIZE = 128
NUM_WORKERS = 2
DATA_DIR = Path('./data')
RESULT_DIR = Path('./results')
RESULT_DIR.mkdir(parents=True, exist_ok=True)

RUN_VIT = False   # 你说不想重训ViT，默认False
RUN_D2NN = True

# D2NN参数（贴近原single-layer文件）
IMG_SIZE = 64
N_pixels = 128
PADDING = (N_pixels - IMG_SIZE) // 2
NUM_LAYERS_D2NN = 12
wl = 700e-9
pixel_size = 2e-6
INIT_DISTANCE = 0.005

D2NN_STAGE1_EPOCHS = 12
D2NN_STAGE2_EPOCHS = 24

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ DEVICE: {DEVICE}')


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [ ]:
# =========================
# 数据集（D2NN输入风格：Resize到64再Pad到128）
# =========================
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Pad([PADDING, PADDING], fill=0, padding_mode='constant'),
])

train_dataset = torchvision.datasets.CIFAR10(str(DATA_DIR), train=True, transform=transform, download=True)
val_dataset = torchvision.datasets.CIFAR10(str(DATA_DIR), train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))

classes = train_dataset.classes
print(f'✅ train={len(train_dataset)}, val={len(val_dataset)}')

In [ ]:
# 可视化几张样本
imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(16, 3))
for i in range(8):
    axes[i].imshow(imgs[i].permute(1,2,0).numpy())
    axes[i].set_title(classes[labels[i].item()], fontsize=9)
    axes[i].axis('off')
plt.suptitle('CIFAR-10样本（Resize+Pad后）')
plt.tight_layout()
plt.show()
print('✅ 样本可视化完成')

In [ ]:
# =========================
# 共享FC头（保持公平对比要求）
# =========================
class SharedFCHead(nn.Module):
    def __init__(self, in_dim, hidden_dim=512, num_classes=10):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.head(x)

In [ ]:
# =========================
# ViT（保留，不默认重训）
# =========================
class ViTWithSharedHead(nn.Module):
    def __init__(self, num_classes=10, fc_hidden_dim=512):
        super().__init__()
        self.backbone = vit_b_16(weights=None)
        in_dim = self.backbone.heads.head.in_features
        self.backbone.heads = nn.Identity()
        self.fc_head = SharedFCHead(in_dim, hidden_dim=fc_hidden_dim, num_classes=num_classes)

    def forward(self, x):
        x = F.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)
        feat = self.backbone(x)
        return self.fc_head(feat)

In [ ]:
# =========================
# D2NN核心结构（严格按single-layer结构思想）
# =========================
class Diffractive_Layer(nn.Module):
    def __init__(self, wl=700e-9, N_pixels=128, pixel_size=2e-6):
        super().__init__()
        fx = torch.fft.fftshift(torch.fft.fftfreq(N_pixels, d=pixel_size))
        fy = torch.fft.fftshift(torch.fft.fftfreq(N_pixels, d=pixel_size))
        fxx, fyy = torch.meshgrid(fx, fy, indexing='ij')

        argument = (2 * np.pi)**2 * ((1. / wl) ** 2 - fxx ** 2 - fyy ** 2)
        tmp = torch.sqrt(torch.abs(argument))
        kz = torch.where(argument >= 0, tmp, 1j * tmp)
        self.register_buffer('kz', kz.to(torch.complex64))

    def forward(self, E, distance):
        fft_c = torch.fft.fft2(E)
        c = torch.fft.fftshift(fft_c, dim=(-2, -1))
        phase = torch.exp(1j * self.kz * distance)
        angular_spectrum = torch.fft.ifft2(torch.fft.ifftshift(c * phase, dim=(-2, -1)))
        return angular_spectrum


class DNN(nn.Module):
    def __init__(self, phase=None, num_layers=12, wl=700e-9, N_pixels=128, pixel_size=2e-6, distance=None):
        super().__init__()
        self.num_layers = num_layers

        self.diffractive_layers = nn.ModuleList([
            Diffractive_Layer(wl=wl, N_pixels=N_pixels, pixel_size=pixel_size) for _ in range(num_layers)
        ])
        self.last_diffractive_layer = Diffractive_Layer(wl=wl, N_pixels=N_pixels, pixel_size=pixel_size)

        # 注册相位参数（风格与single-layer一致）
        for i in range(num_layers):
            self.register_parameter(f'phase_{i}', phase[i])

        # 注册层间距参数（最后一段传播也算一个distance）
        for i in range(num_layers + 1):
            self.register_parameter(f'distance_{i}', distance[i])

        # detector改成共享FC头（按你的要求）
        self.pool = nn.AdaptiveAvgPool2d((8, 8))
        self.fc_head = SharedFCHead(3 * 8 * 8, hidden_dim=512, num_classes=10)

    def forward(self, images):
        # single-layer原始逻辑里是对输入做sqrt转振幅
        E = torch.sqrt(torch.clamp(images, min=0.0) + 1e-8).to(torch.complex64)

        for i, layer in enumerate(self.diffractive_layers):
            d = getattr(self, f'distance_{i}')
            temp = layer(E, d)

            phase = getattr(self, f'phase_{i}')
            constr_phase = 2 * torch.pi * torch.sigmoid(phase)
            exp_j_phase = torch.exp(1j * constr_phase)
            E = temp * exp_j_phase

        d_last = getattr(self, f'distance_{self.num_layers}')
        E = self.last_diffractive_layer(E, d_last)

        Int = torch.abs(E) ** 2
        feat = self.pool(Int).flatten(1)
        logits = self.fc_head(feat)
        return logits, Int

print('✅ DNN结构定义完成（12层 + Shared FC Head）')

In [ ]:
# 初始化相位和层间距参数（保留single-layer思想）
def init_d2nn_params(num_layers=12, N_pixels=128, init_distance=0.005, device='cpu'):
    phase = [
        nn.Parameter(torch.from_numpy(np.random.random(size=(N_pixels, N_pixels)).astype(np.float32)).to(device))
        for _ in range(num_layers)
    ]
    distance = [
        nn.Parameter(torch.tensor(init_distance, dtype=torch.float32, device=device))
        for _ in range(num_layers + 1)
    ]
    return phase, distance

phase, distance = init_d2nn_params(num_layers=NUM_LAYERS_D2NN, N_pixels=N_pixels, init_distance=INIT_DISTANCE, device=DEVICE)
model_d2nn = DNN(phase=phase, num_layers=NUM_LAYERS_D2NN, wl=wl, N_pixels=N_pixels, pixel_size=pixel_size, distance=distance).to(DEVICE)
print('✅ D2NN模型实例化完成')

In [ ]:
# =========================
# 训练工具：evaluate返回tuple（acc, loss, preds, labels）
# =========================
def to_onehot(labels, num_classes=10):
    return F.one_hot(labels, num_classes=num_classes).float()


def evaluate(model, loader, device, criterion):
    model.eval()
    total, correct = 0, 0
    loss_sum = 0.0
    preds_all, labels_all = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Eval', leave=False):
            images = images.to(device)
            labels = labels.to(device)

            out_labels, _ = model(images)
            probs = torch.softmax(out_labels, dim=1)
            target = to_onehot(labels, 10).to(device)
            loss = criterion(probs, target)

            pred = probs.argmax(dim=1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)
            loss_sum += loss.item()

            preds_all.append(pred.detach().cpu())
            labels_all.append(labels.detach().cpu())

    acc = correct / max(total, 1)
    avg_loss = loss_sum / max(len(loader), 1)
    preds_all = torch.cat(preds_all) if preds_all else torch.tensor([])
    labels_all = torch.cat(labels_all) if labels_all else torch.tensor([])
    return acc, avg_loss, preds_all, labels_all


@dataclass
class Hist:
    train_loss: list
    val_loss: list
    val_acc: list

In [ ]:
# 阶段控制：先训相位(+fc_head)，再解锁所有参数（含distance）
def freeze_for_stage1(model):
    for p in model.parameters():
        p.requires_grad = False

    params = []
    for name, p in model.named_parameters():
        if ('phase_' in name) or ('fc_head' in name):
            p.requires_grad = True
            params.append(p)
    return params


def unfreeze_for_stage2(model):
    for p in model.parameters():
        p.requires_grad = True

    phase_params, head_params, dist_params = [], [], []
    for name, p in model.named_parameters():
        if 'phase_' in name:
            phase_params.append(p)
        elif 'distance_' in name:
            dist_params.append(p)
        elif 'fc_head' in name:
            head_params.append(p)
        else:
            head_params.append(p)

    return phase_params, head_params, dist_params

In [ ]:
# =========================
# D2NN两阶段训练（核心）
# =========================
def train_d2nn_two_stage(model, train_loader, val_loader, device,
                         stage1_epochs=12, stage2_epochs=24,
                         stage1_lr=1e-4,
                         stage2_phase_lr=2e-5,
                         stage2_head_lr=1e-5,
                         stage2_dist_lr=1e-6):
    criterion = nn.MSELoss(reduction='sum').to(device)

    best_acc = 0.0
    best_state = None

    hist = Hist(train_loss=[], val_loss=[], val_acc=[])

    # ---------- Stage 1 ----------
    print('
[Stage-1] 先训练相位 + 共享FC头（锁定distance）')
    params_to_update = freeze_for_stage1(model)
    optimizer = torch.optim.Adam(params_to_update, lr=stage1_lr)

    for epoch in range(stage1_epochs):
        model.train()
        ep_loss = 0.0

        for images, labels in tqdm(train_loader, desc=f'S1 {epoch+1}/{stage1_epochs}'):
            images = images.to(device)
            labels = labels.to(device)
            target = to_onehot(labels, 10).to(device)

            out_labels, _ = model(images)
            probs = torch.softmax(out_labels, dim=1)
            loss = criterion(probs, target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            ep_loss += loss.item()

        eval_out = evaluate(model, val_loader, device, criterion)
        val_acc = eval_out[0]   # evaluate返回tuple，第一位是acc
        val_loss = eval_out[1]

        hist.train_loss.append(ep_loss / max(len(train_loader), 1))
        hist.val_loss.append(val_loss)
        hist.val_acc.append(val_acc)

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(f'[S1][{epoch+1}] train_loss={hist.train_loss[-1]:.4f} val_loss={val_loss:.4f} val_acc={val_acc*100:.2f}%')

    # ---------- Stage 2 ----------
    print('
[Stage-2] 解锁所有参数（相位+distance+共享FC头）联合微调')
    phase_params, head_params, dist_params = unfreeze_for_stage2(model)

    # 关键：distance学习率必须显著更小，避免破坏第一阶段学到的相位
    optimizer = torch.optim.Adam([
        {'params': phase_params, 'lr': stage2_phase_lr},
        {'params': head_params, 'lr': stage2_head_lr},
        {'params': dist_params, 'lr': stage2_dist_lr},
    ])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(stage2_epochs, 1))

    for epoch in range(stage2_epochs):
        model.train()
        ep_loss = 0.0

        for images, labels in tqdm(train_loader, desc=f'S2 {epoch+1}/{stage2_epochs}'):
            images = images.to(device)
            labels = labels.to(device)
            target = to_onehot(labels, 10).to(device)

            out_labels, _ = model(images)
            probs = torch.softmax(out_labels, dim=1)
            loss = criterion(probs, target)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss += loss.item()

        scheduler.step()

        eval_out = evaluate(model, val_loader, device, criterion)
        val_acc = eval_out[0]   # evaluate返回tuple，第一位是acc
        val_loss = eval_out[1]

        hist.train_loss.append(ep_loss / max(len(train_loader), 1))
        hist.val_loss.append(val_loss)
        hist.val_acc.append(val_acc)

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(f'[S2][{epoch+1}] train_loss={hist.train_loss[-1]:.4f} val_loss={val_loss:.4f} val_acc={val_acc*100:.2f}%')

    if best_state is not None:
        model.load_state_dict(best_state)

    print(f'✅ D2NN最佳验证准确率: {best_acc*100:.2f}%')
    return model, hist

In [ ]:
# 运行D2NN训练
if RUN_D2NN:
    model_d2nn, d2nn_hist = train_d2nn_two_stage(
        model_d2nn,
        train_loader,
        val_loader,
        DEVICE,
        stage1_epochs=D2NN_STAGE1_EPOCHS,
        stage2_epochs=D2NN_STAGE2_EPOCHS,
        stage1_lr=1e-4,
        stage2_phase_lr=2e-5,
        stage2_head_lr=1e-5,
        stage2_dist_lr=1e-6,
    )
    print('✅ D2NN训练完成')

In [ ]:
# =========================
# 评估与可视化（确保使用最新d2nn_hist，而非旧缓存）
# =========================
criterion_eval = nn.MSELoss(reduction='sum').to(DEVICE)
eval_out = evaluate(model_d2nn, val_loader, DEVICE, criterion_eval)
d2nn_acc = eval_out[0]
d2nn_preds = eval_out[2]
d2nn_labels = eval_out[3]
print(f'✅ D2NN最终验证准确率: {d2nn_acc*100:.2f}%')

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(d2nn_hist.train_loss, marker='o')
plt.title('D2NN 训练损失曲线')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(alpha=0.3)

plt.subplot(1,2,2)
plt.plot(np.array(d2nn_hist.val_acc)*100, marker='o')
plt.title('D2NN 验证准确率曲线')
plt.xlabel('Epoch')
plt.ylabel('Accuracy(%)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('✅ 曲线绘制完成（已使用本次最新训练数据）')

In [ ]:
# 混淆矩阵（中文标题可正常显示）
def build_confusion_matrix(predicted, labels, num_classes=10):
    cm = torch.zeros(num_classes, num_classes, dtype=torch.int64)
    for p, t in zip(predicted, labels):
        cm[t.long(), p.long()] += 1
    return cm


def plot_confusion_matrix(cm, class_names, title='混淆矩阵'):
    cm = cm.numpy()
    plt.figure(figsize=(8,6))
    plt.imshow(cm, cmap='Blues')
    plt.title(title)
    plt.colorbar()
    ticks = np.arange(len(class_names))
    plt.xticks(ticks, class_names, rotation=45, ha='right')
    plt.yticks(ticks, class_names)
    plt.xlabel('预测标签')
    plt.ylabel('真实标签')

    thresh = cm.max()/2 if cm.size > 0 else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i,j]), ha='center', va='center',
                     color='white' if cm[i,j] > thresh else 'black', fontsize=8)
    plt.tight_layout()
    plt.show()

cm = build_confusion_matrix(d2nn_preds, d2nn_labels, num_classes=10)
plot_confusion_matrix(cm, classes, title='D2NN(12层+共享FC头) 混淆矩阵')
print('✅ 混淆矩阵绘制完成')

In [ ]:
# 保存本次结果，避免和旧版本数据混用
result_payload = {
    'd2nn_best_val_acc_est': float(max(d2nn_hist.val_acc) if len(d2nn_hist.val_acc) else 0.0),
    'd2nn_last_val_acc': float(d2nn_hist.val_acc[-1] if len(d2nn_hist.val_acc) else 0.0),
    'stage1_epochs': D2NN_STAGE1_EPOCHS,
    'stage2_epochs': D2NN_STAGE2_EPOCHS,
}
out_path = RESULT_DIR / 'cifar10_d2nn12_shared_fc_v2.json'
out_path.write_text(json.dumps(result_payload, indent=2, ensure_ascii=False), encoding='utf-8')
print('✅ 结果已保存:', out_path)
print(result_payload)

## 调参建议（目标：50%+）

如果你第一次跑还没到 50%，建议按顺序调整：

1. `stage2_epochs` 从 24 提高到 40；
2. 把 `stage2_phase_lr` 提到 `3e-5`，`stage2_head_lr` 保持 `1e-5`；
3. 若震荡明显，把 `stage2_dist_lr` 降到 `5e-7`；
4. 显存允许时把 `BATCH_SIZE` 提高到 200（原仓库常用较大batch）。